## Imports

In [4]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from implicit.als import AlternatingLeastSquares
import faiss

## Configuration

In [5]:
RANDOM_STATE = 42
TOP_K = 10
CANDIDATE_K = 100
FINAL_CANDIDATE_K = 300
SEMANTIC_RETRIEVAL_K = 100
TFIDF_RETRIEVAL_K = 100
ITEM_CF_RETRIEVAL_K = 100
ALS_RETRIEVAL_K = 100
POPULARITY_K = 100
TRENDING_K = 100
COUNTRY_K = 100
RECENT_DAYS = 7
RECENCY_HALF_LIFE_DAYS = 14
MIN_INTERACTIONS_FOR_PERSONALIZATION = 2

## Paths

In [6]:
PROJECT_DIR = Path(r"D:\iPrint-News-Recommendation-Ranking-System")
DATA_DIR = PROJECT_DIR / "Dataset"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
FEATURE_DIR = ARTIFACT_DIR / "features"
INDEX_DIR = ARTIFACT_DIR / "indexes"
SEMANTIC_DIR = ARTIFACT_DIR / "semantic"
CANDIDATE_DIR = ARTIFACT_DIR / "candidates"
for directory in [ARTIFACT_DIR,FEATURE_DIR,INDEX_DIR,SEMANTIC_DIR,CANDIDATE_DIR,]:
    directory.mkdir(parents=True,exist_ok=True)
print("Project directory:", PROJECT_DIR)
print("Candidate directory:", CANDIDATE_DIR)

Project directory: D:\iPrint-News-Recommendation-Ranking-System
Candidate directory: D:\iPrint-News-Recommendation-Ranking-System\artifacts\candidates


## Load Data

In [7]:
consumer = pd.read_csv(DATA_DIR / "consumer_transanctions.csv")
content = pd.read_csv(DATA_DIR / "platform_content.csv")
print("Consumer shape:", consumer.shape)
print("Content shape:", content.shape)

Consumer shape: (72312, 8)
Content shape: (3122, 13)


## Validation

In [8]:
required_consumer_columns = ["event_timestamp","interaction_type","item_id","consumer_id","consumer_session_id","consumer_device_info","consumer_location","country",]
required_content_columns = ["event_timestamp","interaction_type","item_id","producer_id","producer_session_id","producer_device_info","producer_location","producer_country","item_type","item_url","title","text_description","language",]
missing_consumer = [col for col in required_consumer_columns if col not in consumer.columns]
missing_content = [col for col in required_content_columns if col not in content.columns]
assert not missing_consumer, (f"Missing consumer columns: {missing_consumer}")
assert not missing_content, (f"Missing content columns: {missing_content}")
print("Schema validation passed.")

Schema validation passed.


## 1. Normalize IDs

In [9]:
consumer["consumer_id"] = (consumer["consumer_id"].astype(str).str.strip())
consumer["item_id"] = (consumer["item_id"].astype(str).str.strip())
content["item_id"] = (content["item_id"].astype(str).str.strip())
content["producer_id"] = (content["producer_id"].astype(str).str.strip())
consumer["interaction_type"] = (consumer["interaction_type"].astype(str).str.strip().str.lower())
content["interaction_type"] = (content["interaction_type"].astype(str).str.strip().str.lower())
content["language"] = (content["language"].astype(str).str.strip().str.lower())

## 2. Build Latest Content State

In [10]:
content["event_datetime"] = pd.to_datetime(content["event_timestamp"], unit="s", utc=True)

In [11]:
print("Content columns:")
print(content.columns.tolist())
print("Does event_datetime exist?")
print("event_datetime" in content.columns)

Content columns:
['event_timestamp', 'interaction_type', 'item_id', 'producer_id', 'producer_session_id', 'producer_device_info', 'producer_location', 'producer_country', 'item_type', 'item_url', 'title', 'text_description', 'language', 'event_datetime']
Does event_datetime exist?
True


In [12]:
content["event_timestamp"] = pd.to_numeric( content["event_timestamp"], errors="coerce")
content["event_datetime"] = pd.to_datetime(content["event_timestamp"],unit="s",utc=True,errors="coerce")
print("Content event period:", content["event_datetime"].min(), "→", content["event_datetime"].max())
print("Missing event_datetime:", content["event_datetime"].isna().sum())

Content event period: 2016-03-28 19:19:39+00:00 → 2017-02-28 18:51:11+00:00
Missing event_datetime: 0


In [13]:
content_sorted = content.sort_values(["item_id","event_datetime"]).copy()
latest_content_state = (content_sorted.groupby("item_id",as_index=False).tail(1).copy())
latest_content_state["is_available"] = (latest_content_state["interaction_type"].eq("content_present"))
print("Unique articles in content:", latest_content_state["item_id"].nunique())
print("Currently available:", latest_content_state["is_available"].sum())
print("Currently pulled out:",(~latest_content_state[ "is_available"]).sum())

Unique articles in content: 3057
Currently available: 2983
Currently pulled out: 74


In [14]:
required_columns = ["item_id", "event_timestamp", "interaction_type",]
missing_columns = [column for column in required_columns if column not in content.columns]
if missing_columns:
    raise ValueError( f"Missing required content columns: {missing_columns}")
content["item_id"] = (content["item_id"].astype(str).str.strip())
content["interaction_type"] = (content["interaction_type"].astype(str).str.strip().str.lower())
content["event_timestamp"] = pd.to_numeric(content["event_timestamp"],errors="coerce")
content["event_datetime"] = pd.to_datetime(content["event_timestamp"],unit="s",utc=True,errors="coerce")
missing_datetime = (content["event_datetime"].isna().sum())
if missing_datetime > 0:
    print(f"WARNING: {missing_datetime} "f"content rows have invalid timestamps.")
    content = content[content["event_datetime"].notna()].copy()
content_sorted = (content.sort_values(["item_id","event_datetime"]).copy())
latest_content_state = (content_sorted.groupby("item_id",as_index=False).tail(1).copy())
latest_content_state["is_available"] = (latest_content_state["interaction_type"].eq("content_present"))
available_items = set(latest_content_state.loc[latest_content_state["is_available"],"item_id"])
pulled_items = set(latest_content_state.loc[~latest_content_state["is_available"],"item_id"])
print("CONTENT AVAILABILITY SUMMARY")
print("Content lifecycle rows:",len(content))
print("Unique articles:",latest_content_state["item_id"].nunique())
print("Currently available:",len(available_items))
print("Currently pulled out:",len(pulled_items))
print("Available percentage:",round(len(available_items)/latest_content_state["item_id"].nunique() * 100, 2),"%")

CONTENT AVAILABILITY SUMMARY
Content lifecycle rows: 3122
Unique articles: 3057
Currently available: 2983
Currently pulled out: 74
Available percentage: 97.58 %


## 3. Available & Pulled Out Article

In [15]:
available_items = set(latest_content_state.loc[latest_content_state["is_available"],"item_id"])
pulled_items = set(latest_content_state.loc[~latest_content_state["is_available"],"item_id"])
print("Available items:", len(available_items))
print("Pulled-out items:", len(pulled_items))

Available items: 2983
Pulled-out items: 74


## 4. Bulid Article Catalogs

In [16]:
catalog_columns = ["item_id","producer_id","item_type","item_url","title","text_description","language","producer_country","producer_location","event_datetime","is_available",]
article_catalog = latest_content_state[[col for col in catalog_columns if col in latest_content_state.columns]].copy()
article_catalog["title"] = (article_catalog["title"].fillna("").astype(str).str.strip())
article_catalog["text_description"] = (article_catalog["text_description"].fillna("").astype(str).str.strip())
article_catalog = article_catalog.drop_duplicates(subset=["item_id"])
article_catalog = article_catalog.reset_index(drop=True)
print("Article catalog:", article_catalog.shape)

Article catalog: (3057, 11)


## 5. English Available Article Catalogs

In [17]:
english_available_catalog = article_catalog[article_catalog["is_available"].eq(True) & article_catalog["language"].eq("en")].copy()
english_available_catalog = (english_available_catalog.drop_duplicates("item_id").reset_index(drop=True))
print("English available articles:", len(english_available_catalog))

English available articles: 2166


## 6. Interaction Weights

In [18]:
INTERACTION_WEIGHTS = {"content_watched": 1.0,"content_liked": 2.0,"content_saved": 3.0,"content_followed": 4.0,"content_commented_on": 5.0,}
consumer["interaction_weight"] = (consumer["interaction_type"].map(INTERACTION_WEIGHTS).fillna(0.0))
consumer[["interaction_type","interaction_weight"]].drop_duplicates().sort_values("interaction_weight")

,interaction_type,interaction_weight
0,content_watched,1.0
33,content_liked,2.0
25,content_saved,3.0
3,content_followed,4.0
61,content_commented_on,5.0


## 7. Train / Test Split

In [19]:
required_columns = ["consumer_id","event_timestamp","item_id","interaction_type"]
missing_columns = [col for col in required_columns if col not in consumer.columns]
if missing_columns:
    raise ValueError(f"Missing required columns in consumer: {missing_columns}")
consumer["consumer_id"] = (consumer["consumer_id"].astype(str).str.strip())
consumer["item_id"] = (consumer["item_id"].astype(str).str.strip())
consumer["interaction_type"] = (consumer["interaction_type"].astype(str).str.strip().str.lower())
consumer["event_timestamp"] = pd.to_numeric(consumer["event_timestamp"],errors="coerce")
consumer["event_datetime"] = pd.to_datetime(consumer["event_timestamp"],unit="s",utc=True,errors="coerce")
invalid_timestamps = consumer["event_datetime"].isna().sum()
if invalid_timestamps > 0:
    print(f"Removing {invalid_timestamps:,} rows "f"with invalid timestamps.")
    consumer = consumer[consumer["event_datetime"].notna()].copy()
print("Consumer shape:", consumer.shape)
print("Consumer columns:")
print(consumer.columns.tolist())
print("Timestamp range:")
print("Min:", consumer["event_datetime"].min())
print("Max:", consumer["event_datetime"].max())
print("Timestamp dtype:")
print(consumer["event_datetime"].dtype)
print("Missing event_datetime:")
print(consumer["event_datetime"].isna().sum())

Consumer shape: (72312, 10)
Consumer columns:
['event_timestamp', 'interaction_type', 'item_id', 'consumer_id', 'consumer_session_id', 'consumer_device_info', 'consumer_location', 'country', 'interaction_weight', 'event_datetime']
Timestamp range:
Min: 2016-03-14 13:54:36+00:00
Max: 2017-02-28 19:21:51+00:00
Timestamp dtype:
datetime64[ns, UTC]
Missing event_datetime:
0


In [20]:
consumer = consumer.sort_values(["consumer_id","event_datetime"]).reset_index(drop=True)
user_counts = (consumer.groupby("consumer_id").size())
eligible_users = user_counts[user_counts >= MIN_INTERACTIONS_FOR_PERSONALIZATION].index
eligible_consumer = consumer[consumer["consumer_id"].isin(eligible_users)].copy()
test_indices = (eligible_consumer.groupby("consumer_id").tail(1).index)
test_interactions = (eligible_consumer.loc[test_indices].copy())
train_consumer = (eligible_consumer.drop(index=test_indices).copy())
train_consumer = train_consumer.sort_values("event_datetime").reset_index(drop=True)
test_interactions = test_interactions.sort_values("event_datetime").reset_index(drop=True)
print("Eligible users:", len(eligible_users))
print("Train interactions:", len(train_consumer))
print("Test interactions:", len(test_interactions))
print("Train date range:", train_consumer["event_datetime"].min(), "->", train_consumer["event_datetime"].max())
print("Test date range:",test_interactions["event_datetime"].min(), "->", test_interactions["event_datetime"].max())

Eligible users: 1715
Train interactions: 70417
Test interactions: 1715
Train date range: 2016-03-14 13:54:36+00:00 -> 2017-02-28 18:53:25+00:00
Test date range: 2016-03-30 18:37:49+00:00 -> 2017-02-28 19:21:51+00:00


In [21]:
test_truth = (test_interactions.groupby("consumer_id")["item_id"].apply(set).to_dict())
print("Users in test_truth:",len(test_truth))
print("Test items:",sum(len(items) for items in test_truth.values()))

Users in test_truth: 1715
Test items: 1715


## 8. Bulid User Seen Item Lookup

In [22]:
user_seen_items = (train_consumer.groupby("consumer_id")["item_id"].agg(set).to_dict())
print("Users with training history:",len(user_seen_items))

Users with training history: 1715


## 9. Global Popularity

In [23]:
popularity_stats = (train_consumer.groupby("item_id").agg(interaction_count=("item_id","size"),unique_users=("consumer_id","nunique"),weighted_interactions=("interaction_weight","sum"),last_interaction=("event_datetime","max")).reset_index())
popularity_stats = popularity_stats.merge(article_catalog[["item_id","title","language","is_available","producer_id"]],on="item_id",how="left")
popularity_stats = popularity_stats[popularity_stats["is_available"].eq(True)]
popularity_stats = popularity_stats.sort_values(["weighted_interactions","unique_users","interaction_count"],ascending=False).reset_index(drop=True)
print("Popularity catalog:",popularity_stats.shape)

Popularity catalog: (2911, 9)


## 10. Popularity Score

In [24]:
def min_max_scale(series):
    series = series.astype(float)
    min_value = series.min()
    max_value = series.max()
    if pd.isna(min_value) or max_value == min_value:
        return pd.Series(np.ones(len(series)), index=series.index)
    return ((series - min_value) / (max_value - min_value))
popularity_stats["popularity_score"] = (min_max_scale(np.log1p(popularity_stats["weighted_interactions"])))

## 11. Popularity Candidate Generator

In [25]:
def generate_popularity_candidates(user_id,retrieval_k=POPULARITY_K,history_df=train_consumer):
    seen_items = get_seen_items(user_id,history_df)
    candidates = popularity_stats[~popularity_stats["item_id"].isin(seen_items)].head(retrieval_k).copy()
    if candidates.empty:
        return pd.DataFrame(columns=["consumer_id","item_id","popularity_score"])
    candidates["consumer_id"] = user_id
    return candidates[["consumer_id","item_id","popularity_score"]].reset_index(drop=True)

## 12. Trending Candidates

In [26]:
reference_time = train_consumer["event_datetime"].max()
recent_cutoff = (reference_time - pd.Timedelta(days=RECENT_DAYS))
recent_interactions = train_consumer[train_consumer["event_datetime"] >= recent_cutoff].copy()
trending_stats = (recent_interactions.groupby("item_id").agg(recent_interactions=("item_id","size"),recent_unique_users=("consumer_id","nunique"),recent_weighted_interactions=("interaction_weight","sum"),latest_interaction=("event_datetime","max")).reset_index())
trending_stats = trending_stats.merge(article_catalog[["item_id","title","language","is_available"]],on="item_id",how="left")
trending_stats = trending_stats[trending_stats["is_available"].eq(True)].copy()

## 13. Recency Adjusted Treading Score

In [27]:
def recency_weight(event_time,reference_time,half_life_days=RECENCY_HALF_LIFE_DAYS):
    age_days = (reference_time - event_time).total_seconds() / 86400.0
    age_days = max(age_days, 0.0)
    return 0.5 ** (age_days / half_life_days)
recent_interactions["recency_weight"] = (recent_interactions["event_datetime"].apply(lambda x: recency_weight(x,reference_time)))
recent_interactions["recency_weighted_signal"] = (recent_interactions["interaction_weight"] * recent_interactions["recency_weight"])
trending_stats = (recent_interactions.groupby("item_id").agg(recent_interactions=("item_id","size"),recent_unique_users=("consumer_id","nunique"),trending_score=("recency_weighted_signal","sum"),latest_interaction=("event_datetime","max")).reset_index())
trending_stats = trending_stats.merge(article_catalog[["item_id","title","language","is_available"]],on="item_id",how="left")
trending_stats = trending_stats[trending_stats["is_available"].eq(True)].copy()
trending_stats["trending_score"] = min_max_scale(np.log1p(trending_stats["trending_score"]))
trending_stats = trending_stats.sort_values("trending_score",ascending=False).reset_index(drop=True)

## 14. Trending Candidate Generator

In [28]:
def generate_trending_candidates(user_id,retrieval_k=TRENDING_K):
    seen_items = get_seen_items(user_id)
    candidates = trending_stats[~trending_stats["item_id"].isin(seen_items)].head(retrieval_k).copy()
    if candidates.empty:
        return pd.DataFrame(columns=["consumer_id","item_id","trending_score"])
    candidates["consumer_id"] = user_id
    return candidates[["consumer_id","item_id","trending_score"]].reset_index(drop=True)

## 15. Country Popularity

In [29]:
user_country = (train_consumer.sort_values("event_datetime").groupby("consumer_id")["country"].agg(lambda x: (x.dropna().iloc[-1] if not x.dropna().empty else None)).to_dict())
country_popularity = (train_consumer.dropna(subset=["country"]).groupby(["country", "item_id"]).agg(country_interactions=("item_id","size"),country_unique_users=("consumer_id","nunique"),country_weighted_interactions=("interaction_weight","sum")).reset_index())
country_popularity = country_popularity.merge(article_catalog[["item_id","title","language","is_available"]],on="item_id",how="left")
country_popularity = country_popularity[country_popularity["is_available"].eq(True)]
country_popularity["country_score"] = (np.log1p(country_popularity["country_weighted_interactions"]))
country_popularity = (country_popularity.sort_values(["country","country_score"], ascending=False))

## 16. Country Candidate Generator

In [30]:
def generate_country_candidates(user_id,retrieval_k=COUNTRY_K):
    country = user_country.get(str(user_id))
    seen_items = get_seen_items(user_id)
    if country is None:
        return pd.DataFrame(columns=["consumer_id","item_id","country_score"])
    candidates = country_popularity[country_popularity["country"].eq(country) & ~country_popularity["item_id"].isin(seen_items)].head(retrieval_k).copy()
    if candidates.empty:
        return pd.DataFrame(columns=["consumer_id","item_id","country_score"])
    candidates["country_score"] = min_max_scale(candidates["country_score"])
    candidates["consumer_id"] = user_id
    return candidates[["consumer_id","item_id","country_score"]].reset_index(drop=True)

## 17. TF & IDF Catalogs

In [31]:
tfidf_catalog = (english_available_catalog.copy())
tfidf_catalog["article_text"] = ("Title: " + tfidf_catalog["title"] + " Description: " + tfidf_catalog["text_description"])
tfidf_vectorizer = TfidfVectorizer(max_features=10000,stop_words="english",ngram_range=(1, 2),min_df=2)
tfidf_matrix = tfidf_vectorizer.fit_transform(tfidf_catalog["article_text"])
tfidf_matrix = normalize(tfidf_matrix,norm="l2",axis=1)
tfidf_item_to_index = {item_id: index for index, item_id in enumerate(tfidf_catalog["item_id"])}
tfidf_index_to_item = {index: item_id for item_id, index in tfidf_item_to_index.items()}
print("TF-IDF matrix:", tfidf_matrix.shape)

TF-IDF matrix: (2166, 10000)


## 18. Bulid User TF & IDF Profile

In [32]:
def build_user_tfidf_profile(user_id,reference_time,history_df=train_consumer):
    history = history_df[(history_df["consumer_id"] == str(user_id)) & (history_df["event_datetime"] < reference_time)].copy()
    history = history[history["item_id"].isin(tfidf_item_to_index)]
    if history.empty:
        return None
    vectors = []
    weights = []
    for _, row in history.iterrows():
        item_id = row["item_id"]
        index = tfidf_item_to_index.get(item_id)
        if index is None:
            continue
        vector = tfidf_matrix[index]
        recency = recency_weight(row["event_datetime"],reference_time)
        weight = (row["interaction_weight"] * recency)
        if weight <= 0:
            continue
        vectors.append(vector)
        weights.append(weight)
    if not vectors:
        return None
    profile = vectors[0] * weights[0]
    for vector, weight in zip(vectors[1:],weights[1:]):
        profile += vector * weight
    profile = normalize(profile,norm="l2")
    return profile

## 19. TF & IDF Candidate Generator

In [33]:
def generate_tfidf_candidates(user_id,retrieval_k=TFIDF_RETRIEVAL_K,reference_time=None):
    if reference_time is None:
        reference_time = train_consumer["event_datetime"].max()
    user_profile = build_user_tfidf_profile(user_id=user_id,reference_time=reference_time,history_df=train_consumer)
    if user_profile is None:
        return pd.DataFrame(columns=["consumer_id","item_id","tfidf_score"])
    scores = (user_profile @ tfidf_matrix.T).toarray().ravel()
    ranking = np.argsort(-scores)
    seen_items = get_seen_items(user_id)
    rows = []
    for index in ranking:
        item_id = tfidf_index_to_item[int(index)]
        if item_id in seen_items:
            continue
        rows.append({"consumer_id": user_id,"item_id": item_id,"tfidf_score": float(scores[index])})
        if len(rows) >= retrieval_k:
            break
    return pd.DataFrame(rows)

## 20. Item CF

In [34]:
cf_interactions = (train_consumer.groupby(["consumer_id","item_id"], as_index=False).agg(interaction_strength=("interaction_weight","sum")))
cf_users = (cf_interactions["consumer_id"].drop_duplicates().tolist())
cf_items = (cf_interactions["item_id"].drop_duplicates().tolist())
cf_user_to_index = {user_id: index for index, user_id in enumerate(cf_users)}
cf_item_to_index = {item_id: index for index, item_id in enumerate(cf_items)}
cf_index_to_item = {index: item_id for item_id, index in cf_item_to_index.items()}
rows = [cf_user_to_index[user_id] for user_id in cf_interactions["consumer_id"]]
cols = [cf_item_to_index[item_id] for item_id in cf_interactions["item_id"]]
values = (cf_interactions["interaction_strength"].astype(np.float32))
user_item_matrix = csr_matrix((values,(rows, cols)),shape=(len(cf_users),len(cf_items)))
print("User-item matrix:",user_item_matrix.shape)

User-item matrix: (1715, 2980)


## 21. Item Item Cosine Similarity

In [35]:
from sklearn.metrics.pairwise import cosine_similarity

In [36]:
item_user_matrix = user_item_matrix.T.tocsr()
item_similarity = cosine_similarity(item_user_matrix)
print("Item similarity shape:", item_similarity.shape)

Item similarity shape: (2980, 2980)


## 22. Item CF Candidate Generator

In [37]:
def generate_item_cf_candidates(user_id,retrieval_k=ITEM_CF_RETRIEVAL_K):
    user_id = str(user_id)
    if user_id not in cf_user_to_index:
        return pd.DataFrame(columns=["consumer_id","item_id","cf_score"])
    user_index = cf_user_to_index[user_id]
    user_row = (user_item_matrix[user_index].toarray().ravel())
    interacted_indices = np.flatnonzero(user_row > 0)
    if len(interacted_indices) == 0:
        return pd.DataFrame(columns=["consumer_id","item_id","cf_score"])
    scores = np.zeros(len(cf_items),dtype=np.float32)
    for item_index in interacted_indices:
        interaction_strength = (user_row[item_index])
        scores += (item_similarity[item_index] * interaction_strength)
    seen_items = get_seen_items(user_id)
    ranking = np.argsort(-scores)
    rows = []
    for index in ranking:
        if scores[index] <= 0:
            continue
        item_id = cf_index_to_item[int(index)]
        if item_id in seen_items:
            continue
        if item_id not in available_items:
            continue
        rows.append({"consumer_id": user_id,"item_id": item_id,"cf_score": float(scores[index])})
        if len(rows) >= retrieval_k:
            break
    return pd.DataFrame(rows)

## 23. ALS Matrix

In [38]:
als_user_to_index = cf_user_to_index.copy()
als_item_to_index = cf_item_to_index.copy()
als_index_to_item = cf_index_to_item.copy()
als_matrix = user_item_matrix.astype( np.float32)
print("ALS matrix:", als_matrix.shape)

ALS matrix: (1715, 2980)


## 24. Train ALS

In [39]:
als_model = AlternatingLeastSquares(factors=64,regularization=0.05,iterations=20,random_state=RANDOM_STATE)
als_model.fit(als_matrix)
print("ALS training complete.")

100%|██████████| 20/20 [00:00<00:00, 76.33it/s]

ALS training complete.


## 25. ALS Candidate Generator

In [40]:
def generate_als_candidates(user_id,retrieval_k=ALS_RETRIEVAL_K):
    user_id = str(user_id)
    if user_id not in als_user_to_index:
        return pd.DataFrame(columns=["consumer_id","item_id","als_score"])
    user_index = als_user_to_index[user_id]
    try:
        item_indices, scores = (als_model.recommend(userid=user_index,user_items=als_matrix[user_index],N=retrieval_k * 3,filter_already_liked_items=True))
    except Exception as exc:
        print(f"ALS recommendation failed for "f"user={user_id}: {exc}")
        return pd.DataFrame(columns=["consumer_id","item_id","als_score"])
    rows = []
    for index, score in zip(item_indices,scores):
        item_id = als_index_to_item[int(index)]
        if item_id not in available_items:
            continue
        rows.append({"consumer_id": user_id,"item_id": item_id,"als_score": float(score)})
        if len(rows) >= retrieval_k:
            break
    return pd.DataFrame(rows)

## 26. Load Semantic Embeddings

In [41]:
ARTICLE_EMBEDDINGS_PATH = (FEATURE_DIR / "article_embeddings.npy")
ARTICLE_EMBEDDING_INDEX_PATH = (FEATURE_DIR / "article_embedding_index.parquet")
FAISS_INDEX_PATH = (INDEX_DIR / "faiss" / "article_cosine.index")
if (ARTICLE_EMBEDDINGS_PATH.exists() and ARTICLE_EMBEDDING_INDEX_PATH.exists() and FAISS_INDEX_PATH.exists()):
    article_embeddings = np.load(ARTICLE_EMBEDDINGS_PATH)
    embedding_metadata = pd.read_parquet(ARTICLE_EMBEDDING_INDEX_PATH)
    faiss_index = faiss.read_index(str(FAISS_INDEX_PATH))
    print("Loaded semantic embeddings:", article_embeddings.shape)
    print("Loaded FAISS index:",faiss_index.ntotal)
else:
    print("Semantic artifacts not found.")

Loaded semantic embeddings: (2166, 384)
Loaded FAISS index: 2166


## 27. Semantic Mappings

In [42]:
item_to_embedding_index = {item_id: int(index) for item_id, index in zip(embedding_metadata["item_id"], embedding_metadata["embedding_index"])}
index_to_item = {int(index): item_id for item_id, index in zip(embedding_metadata["item_id"],embedding_metadata["embedding_index"])}
semantic_available_items = set(embedding_metadata["item_id"])
print("Semantic items:", len(semantic_available_items))

Semantic items: 2166


## 28. Recency Weighted Semantic User Embeddings

In [43]:
def build_user_embedding_at_time(user_id, cutoff_time,history_df=train_consumer):
    user_id = str(user_id)
    history = history_df[(history_df["consumer_id"] == user_id) & (history_df["event_datetime"] < cutoff_time)].copy()
    history = history[history["item_id"].isin(item_to_embedding_index)]
    if history.empty:
        return None
    weighted_vectors = []
    total_weight = 0.0
    for _, row in history.iterrows():
        item_id = row["item_id"]
        embedding_index = (item_to_embedding_index.get(item_id))
        if embedding_index is None:
            continue
        interaction_weight = (row["interaction_weight"])
        recency = recency_weight(row["event_datetime"],cutoff_time)
        weight = (interaction_weight  * recency)
        if weight <= 0:
            continue
        vector = article_embeddings[embedding_index].astype(np.float32)
        weighted_vectors.append(vector * weight)
        total_weight += weight
    if not weighted_vectors:
        return None
    user_vector = np.sum(weighted_vectors,axis=0)
    if total_weight <= 0:
        return None
    user_vector /= total_weight
    norm = np.linalg.norm(user_vector)
    if norm == 0:
        return None
    user_vector /= norm
    return user_vector.astype(np.float32)

## 29. Semantic Candidate Generator

In [44]:
def generate_semantic_candidates(user_id,retrieval_k=SEMANTIC_RETRIEVAL_K,cutoff_time=None,history_df=train_consumer):
    if cutoff_time is None:
        cutoff_time = history_df["event_datetime"].max()
    user_vector = build_user_embedding_at_time(user_id=user_id,cutoff_time=cutoff_time,history_df=history_df)
    if user_vector is None:
        return pd.DataFrame(columns=["consumer_id","item_id","semantic_score"])
    search_k = min(retrieval_k * 5,faiss_index.ntotal)
    scores, indices = (faiss_index.search(user_vector.reshape(1, -1).astype(np.float32),search_k))
    seen_items = get_seen_items(user_id)
    rows = []
    for score, index in zip(scores[0],indices[0]):
        if index < 0:
            continue
        item_id = index_to_item[int(index)]
        if item_id in seen_items:
            continue
        if item_id not in available_items:
            continue
        if item_id not in semantic_available_items:
            continue
        rows.append({"consumer_id": user_id,"item_id": item_id,"semantic_score": float(score)})
        if len(rows) >= retrieval_k:
            break
    return pd.DataFrame(rows)

## 30. Validate Each Candidate Generator

In [45]:
def get_seen_items(user_id,history_df=train_consumer):
    if history_df is None or history_df.empty:
        return set()
    user_history = history_df[history_df["consumer_id"] == user_id]
    return set(user_history["item_id"].dropna().astype(str).str.strip())

In [46]:
sample_user = next(iter(test_truth.keys()))
seen_items = get_seen_items(user_id=sample_user,history_df=train_consumer)
print("Sample user:", sample_user)
print("Seen articles:", len(seen_items))
print("First 10 seen items:", list(seen_items)[:10])

Sample user: -1007001694607905623
Seen articles: 5
First 10 seen items: ['-6623581327558800021', '1469580151036142903', '-793729620925729327', '7270966256391553686', '-5065077552540450930']


In [47]:
consumer["consumer_id"] = (consumer["consumer_id"].astype(str).str.strip())
consumer["item_id"] = (consumer["item_id"].astype(str).str.strip())
train_consumer["consumer_id"] = (train_consumer["consumer_id"].astype(str).str.strip())
train_consumer["item_id"] = (train_consumer["item_id"].astype(str).str.strip())
test_interactions["consumer_id"] = (test_interactions["consumer_id"].astype(str).str.strip())
test_interactions["item_id"] = (test_interactions["item_id"].astype(str).str.strip())
content["item_id"] = (content["item_id"].astype(str).str.strip())
print("ID normalization complete.")

ID normalization complete.


In [48]:
sample_user = next(iter(test_truth.keys()))
print("Sample user:", sample_user)
pop_test = generate_popularity_candidates(sample_user, retrieval_k=10)
trend_test = generate_trending_candidates(sample_user,retrieval_k=10)
country_test = generate_country_candidates(sample_user,retrieval_k=10)
tfidf_test = generate_tfidf_candidates(sample_user,retrieval_k=10)
cf_test = generate_item_cf_candidates(sample_user,retrieval_k=10)
als_test = generate_als_candidates(sample_user,retrieval_k=10)
semantic_test = generate_semantic_candidates(sample_user,retrieval_k=10)
print("Popularity:", len(pop_test))
print("Trending:", len(trend_test))
print("Country:", len(country_test))
print("TF-IDF:", len(tfidf_test))
print("Item-CF:", len(cf_test))
print("ALS:", len(als_test))
print("Semantic:", len(semantic_test))

Sample user: -1007001694607905623
Popularity: 10
Trending: 10
Country: 10
TF-IDF: 10
Item-CF: 10
ALS: 10
Semantic: 0


## 31. Candidate Source Registry

In [49]:
CANDIDATE_GENERATORS = {"popularity": generate_popularity_candidates,"trending": generate_trending_candidates,"country": generate_country_candidates,"tfidf": generate_tfidf_candidates,"item_cf": generate_item_cf_candidates,"als": generate_als_candidates,"semantic": generate_semantic_candidates,}

## 32. Generate Candidates For One User

In [50]:
def generate_all_candidates_for_user(user_id,retrieval_k=CANDIDATE_K):
    candidate_frames = []
    generators = {"popularity": generate_popularity_candidates,"trending": generate_trending_candidates,"country": generate_country_candidates,"tfidf": generate_tfidf_candidates,"item_cf": generate_item_cf_candidates,"als": generate_als_candidates,"semantic": generate_semantic_candidates,}
    for source_name, generator in generators.items():
        try:
            candidates = generator(user_id=user_id, retrieval_k=retrieval_k)
        except Exception as exc:
            print(f"[WARNING] " f"{source_name} failed " f"for user {user_id}: {exc}")
            continue
        if candidates.empty:
            continue
        candidates = candidates.copy()
        candidates["source"] = (source_name)
        candidate_frames.append(candidates)
    if not candidate_frames:
        return pd.DataFrame(columns=["consumer_id","item_id","source"])
    return pd.concat(candidate_frames,ignore_index=True)

## 33. Generate Candidates For All Test Users

In [51]:
all_candidate_frames = []
test_users = list(test_truth.keys())
print("Generating candidates for", len(test_users), "users...")
for counter, user_id in enumerate(test_users, start=1):
    candidates = (generate_all_candidates_for_user(user_id=user_id, retrieval_k=CANDIDATE_K))
    if not candidates.empty:
        all_candidate_frames.append(candidates)
    if counter % 100 == 0:
        print(f"Processed {counter}/" f"{len(test_users)} users")
if all_candidate_frames:
    raw_candidates = pd.concat(all_candidate_frames, ignore_index=True)
else:
    raw_candidates = pd.DataFrame(columns=["consumer_id","item_id","source"])
print("Raw candidate rows:", len(raw_candidates))

Generating candidates for 1715 users...
Processed 100/1715 users
Processed 200/1715 users
Processed 300/1715 users
Processed 400/1715 users
Processed 500/1715 users
Processed 600/1715 users
Processed 700/1715 users
Processed 800/1715 users
Processed 900/1715 users
Processed 1000/1715 users
Processed 1100/1715 users
Processed 1200/1715 users
Processed 1300/1715 users
Processed 1400/1715 users
Processed 1500/1715 users
Processed 1600/1715 users
Processed 1700/1715 users
Raw candidate rows: 924719


## 34. Inspect Source Coverages

In [52]:
source_coverage = (raw_candidates.groupby("source").agg(candidate_rows=("item_id","size"),users=("consumer_id","nunique"),unique_items=("item_id","nunique")).sort_values("candidate_rows",ascending=False))
display(source_coverage)

,candidate_rows,users,unique_items
source,,,
als,171500,1715,2277
popularity,171500,1715,191
item_cf,171182,1714,2884
country,169212,1700,511
tfidf,148100,1481,2159
trending,93225,1715,55


## 35. Convert Source Rows Into Wide Candidate Features

In [53]:
score_columns = ["popularity_score","trending_score","country_score","tfidf_score","cf_score","als_score","semantic_score",]
available_score_columns = [col for col in score_columns if col in raw_candidates.columns]
candidate_scores = (raw_candidates.groupby(["consumer_id","item_id"],as_index=False)[available_score_columns].max())
print("Unique user-item candidates:", len(candidate_scores))

Unique user-item candidates: 666597


In [54]:
source_presence = (raw_candidates[["consumer_id","item_id","source"]].drop_duplicates().assign(present=1).pivot_table(index=["consumer_id","item_id"],columns="source",values="present",fill_value=0).reset_index())
source_presence.columns.name = None
source_presence = source_presence.rename(columns={"popularity": "from_popularity","trending": "from_trending","country": "from_country","tfidf": "from_tfidf","item_cf": "from_item_cf","als":
 "from_als", "semantic": "from_semantic"})
candidate_scores = candidate_scores.merge(source_presence,on=["consumer_id","item_id"],how="left")

## 36. Fill Missing Source Scores

In [55]:
for column in score_columns:
    if column not in candidate_scores.columns:
        candidate_scores[column] = 0.0
    candidate_scores[column] = (candidate_scores[column].fillna(0.0).astype(float))
source_flags = ["from_popularity","from_trending","from_country","from_tfidf","from_item_cf","from_als","from_semantic",]
for column in source_flags:
    if column not in candidate_scores.columns:
        candidate_scores[column] = 0
    candidate_scores[column] = (candidate_scores[column].fillna(0).astype(int))

## 37. Number Of Candidates Sources

In [56]:
candidate_scores["num_sources"] = (candidate_scores[source_flags].sum(axis=1))
candidate_scores["multi_source_candidate"] = (candidate_scores["num_sources"] >= 2).astype(int)

## 38. Filtering : Seen Articles & Unavailable Articles

In [57]:
before_seen_filter = len(candidate_scores)
candidate_scores = candidate_scores[candidate_scores.apply(lambda row: (row["item_id"] not in get_seen_items(row["consumer_id"])),axis=1)].copy()
after_seen_filter = len(candidate_scores)
print("Removed seen candidates:", before_seen_filter - after_seen_filter)

Removed seen candidates: 0


In [58]:
before_availability_filter = len(candidate_scores)
candidate_scores = candidate_scores[candidate_scores["item_id"].isin(available_items)].copy()
after_availability_filter = len(candidate_scores)
print("Removed unavailable candidates:", before_availability_filter - after_availability_filter)

Removed unavailable candidates: 0


## 39. Join Article Metadata

In [59]:
article_metadata = article_catalog[["item_id","producer_id","item_type","title","text_description","language","item_url","producer_country","producer_location","event_datetime","is_available",]].copy()
candidate_scores = candidate_scores.merge(article_metadata,on="item_id",how="left")
print("Candidate table:", candidate_scores.shape)

Candidate table: (666597, 28)


## 39. Article Freshness

In [60]:
print("train_consumer" in globals())
print("consumer" in globals())

if "consumer" in globals():
    print("consumer shape:", consumer.shape)
    print("consumer columns:", consumer.columns.tolist())

True
True
consumer shape: (72312, 10)
consumer columns: ['event_timestamp', 'interaction_type', 'item_id', 'consumer_id', 'consumer_session_id', 'consumer_device_info', 'consumer_location', 'country', 'interaction_weight', 'event_datetime']


In [61]:
consumer["event_timestamp"] = pd.to_numeric(consumer["event_timestamp"],errors="coerce")
consumer["event_datetime"] = pd.to_datetime(consumer["event_timestamp"],unit="s",utc=True,errors="coerce")
consumer = consumer[consumer["event_datetime"].notna()].copy()
print("Consumer shape:", consumer.shape)
print("Datetime column:", consumer["event_datetime"].dtype)

Consumer shape: (72312, 10)
Datetime column: datetime64[ns, UTC]


In [65]:
consumer = consumer.sort_values(["consumer_id", "event_datetime"]).reset_index(drop=True)
user_counts = (consumer.groupby("consumer_id").size())
eligible_users = user_counts[user_counts >= MIN_INTERACTIONS_FOR_PERSONALIZATION].index
eligible_consumer = consumer[consumer["consumer_id"].isin(eligible_users)].copy()
test_indices = (eligible_consumer.groupby("consumer_id").tail(1).index)
test_interactions = (eligible_consumer.loc[test_indices].copy())
train_consumer = (eligible_consumer.drop(index=test_indices).copy())
train_consumer = (train_consumer.sort_values("event_datetime").reset_index(drop=True))
test_interactions = (test_interactions.sort_values("event_datetime").reset_index(drop=True))
print("Eligible users:", len(eligible_users))
print("Train interactions:", len(train_consumer))
print("Test interactions:", len(test_interactions))
print(train_consumer.shape)
print(train_consumer["event_datetime"].min())
print(train_consumer["event_datetime"].max())

Eligible users: 1715
Train interactions: 70417
Test interactions: 1715
(70417, 10)
2016-03-14 13:54:36+00:00
2017-02-28 18:53:25+00:00


In [66]:
reference_time = train_consumer["event_datetime"].max()
candidate_scores["article_age_days"] = ((reference_time - candidate_scores["event_datetime"]).dt.total_seconds() / 86400.0)
candidate_scores["article_age_days"] = (candidate_scores["article_age_days"].clip(lower=0))
candidate_scores["freshness_score"] = (0.5 ** (candidate_scores["article_age_days"] / RECENCY_HALF_LIFE_DAYS))
print("Reference time:", reference_time)
print(candidate_scores[["item_id","event_datetime","article_age_days","freshness_score"]].head())

Reference time: 2017-02-28 18:53:25+00:00
                item_id            event_datetime  article_age_days  \
0     -1038011342017850 2016-05-17 16:50:00+00:00        287.085706   
1  -1101361754763388054 2016-12-07 12:42:59+00:00         83.257245   
2   -115909536143817330 2017-02-03 13:18:04+00:00         25.232882   
3  -1199490911632553070 2016-06-29 11:47:16+00:00        244.295938   
4  -1254906787526072320 2016-06-02 16:11:44+00:00        271.112280   

   freshness_score  
0     6.714941e-07  
1     1.621029e-02  
2     2.867074e-01  
3     5.586167e-06  
4     1.480830e-06  


## 40. Global Article Statistics

In [69]:
article_stats = (train_consumer.groupby("item_id").agg(total_interactions=("item_id","size"),unique_users=("consumer_id","nunique"),weighted_interactions=("interaction_weight","sum"),last_user_interaction=("event_datetime","max")).reset_index())
candidate_scores = candidate_scores.merge(article_stats,on="item_id",how="left")
candidate_scores[["total_interactions","unique_users","weighted_interactions"]] = candidate_scores[["total_interactions","unique_users","weighted_interactions"]].fillna(0)

## 41. Popularity Normalized Features

In [71]:
candidate_scores["log_total_interactions"] = np.log1p(candidate_scores["total_interactions"])
candidate_scores["log_unique_users"] = np.log1p(candidate_scores["unique_users"])
candidate_scores["log_weighted_interactions"] = np.log1p(candidate_scores["weighted_interactions"])

## 42. User Level Features

In [73]:
user_stats = (train_consumer.groupby("consumer_id").agg(user_interactions=("item_id","size"),user_unique_items=("item_id","nunique"),user_unique_sessions=("consumer_session_id","nunique"),user_total_weight=("interaction_weight","sum"),user_last_activity=("event_datetime","max")).reset_index())
candidate_scores = candidate_scores.merge(user_stats,on="consumer_id",how="left")

In [77]:
user_activity_features = (train_consumer.groupby("consumer_id").agg(user_last_activity=("event_datetime", "max"),user_total_interactions=("item_id", "size"),user_unique_articles=("item_id", "nunique")).reset_index())
print("User activity features:")
print(user_activity_features.head())
print(user_activity_features.shape)

User activity features:
            consumer_id        user_last_activity  user_total_interactions  \
0  -1007001694607905623 2017-02-16 10:14:40+00:00                        6   
1  -1032019229384696495 2017-02-17 19:58:54+00:00                     1884   
2   -108842214936804958 2017-02-16 14:05:21+00:00                      513   
3  -1093393486211919385 2016-09-28 12:40:19+00:00                        1   
4  -1110220372195277179 2016-07-11 15:17:52+00:00                        5   

   user_unique_articles  
0                     5  
1                   648  
2                   270  
3                     1  
4                     3  
(1715, 4)


In [78]:
candidate_scores = candidate_scores.merge(user_activity_features,on="consumer_id",how="left")
print("candidate_scores shape:", candidate_scores.shape)
print(candidate_scores[["consumer_id", "user_last_activity", "user_total_interactions","user_unique_articles"]].head())
print("user_last_activity" in candidate_scores.columns)

candidate_scores shape: (666597, 58)
            consumer_id        user_last_activity  user_total_interactions  \
0  -1007001694607905623 2017-02-16 10:14:40+00:00                        6   
1  -1007001694607905623 2017-02-16 10:14:40+00:00                        6   
2  -1007001694607905623 2017-02-16 10:14:40+00:00                        6   
3  -1007001694607905623 2017-02-16 10:14:40+00:00                        6   
4  -1007001694607905623 2017-02-16 10:14:40+00:00                        6   

   user_unique_articles  
0                     5  
1                     5  
2                     5  
3                     5  
4                     5  
True


In [80]:
candidate_scores["user_inactivity_days"] = ((reference_time - candidate_scores["user_last_activity"]).dt.total_seconds() / 86400.0).clip(lower=0)
candidate_scores["user_activity_recency"] = (0.5 ** (candidate_scores["user_inactivity_days"] / RECENCY_HALF_LIFE_DAYS))
print(candidate_scores[["consumer_id", "user_last_activity", "user_inactivity_days","user_activity_recency"]].head())

            consumer_id        user_last_activity  user_inactivity_days  \
0  -1007001694607905623 2017-02-16 10:14:40+00:00             12.360243   
1  -1007001694607905623 2017-02-16 10:14:40+00:00             12.360243   
2  -1007001694607905623 2017-02-16 10:14:40+00:00             12.360243   
3  -1007001694607905623 2017-02-16 10:14:40+00:00             12.360243   
4  -1007001694607905623 2017-02-16 10:14:40+00:00             12.360243   

   user_activity_recency  
0               0.542286  
1               0.542286  
2               0.542286  
3               0.542286  
4               0.542286  


In [82]:
candidate_scores["user_last_activity"] = (pd.to_datetime(candidate_scores["user_last_activity"],utc=True,errors="coerce"))
candidate_scores["user_inactivity_days"] = ((reference_time - candidate_scores["user_last_activity"]).dt.total_seconds() / 86400.0)
candidate_scores["user_inactivity_days"] = (candidate_scores["user_inactivity_days"].fillna(365.0).clip(lower=0))
candidate_scores["user_activity_recency"] = (0.5 ** (candidate_scores["user_inactivity_days"] / RECENCY_HALF_LIFE_DAYS))

In [84]:
candidate_scores["user_country"] = (candidate_scores["consumer_id"].map(user_country))
candidate_scores["country_match"] = (candidate_scores["user_country"] == candidate_scores["producer_country"]).fillna(False).astype(int)

In [87]:
user_producer_affinity = (train_consumer.merge(article_catalog[["item_id","producer_id"]], on="item_id", how="left").groupby(["consumer_id","producer_id"]).agg(producer_interactions=("item_id","size"),producer_weight=("interaction_weight","sum")).reset_index())
candidate_scores = candidate_scores.merge(user_producer_affinity,on=["consumer_id","producer_id"],how="left")
candidate_scores["producer_interactions"] = candidate_scores["producer_interactions"].fillna(0)
candidate_scores["producer_weight"] = candidate_scores["producer_weight"].fillna(0)

## 43. Session Affinity

In [88]:
recent_session_cutoff = (reference_time - pd.Timedelta(hours=24))
recent_session_history = train_consumer[train_consumer["event_datetime"] >= recent_session_cutoff].copy()
recent_session_items = (recent_session_history.groupby("consumer_id")["item_id"].agg(set).to_dict())
candidate_scores["recent_session_related"] = candidate_scores.apply(lambda row: int(row["item_id"] in recent_session_items.get(row["consumer_id"],set())),axis=1)

## 44. Candidate Rank By Source

In [90]:
def add_source_rank(candidates, score_column,rank_column):
    if candidates.empty:
        return candidates
    candidates = candidates.copy()
    candidates[rank_column] = (candidates.groupby("consumer_id")[score_column].rank(method="first", ascending=False))
    return candidates

In [91]:
def generate_source_candidates(user_id, retrieval_k=CANDIDATE_K):
    output = []
    generators = {"popularity": generate_popularity_candidates, "trending": generate_trending_candidates,"country": generate_country_candidates, "tfidf": generate_tfidf_candidates, "item_cf": generate_item_cf_candidates, "als": generate_als_candidates, "semantic": generate_semantic_candidates}
    score_mapping = {"popularity": "popularity_score","trending": "trending_score", "country": "country_score", "tfidf": "tfidf_score", "item_cf": "cf_score", "als": "als_score","semantic": "semantic_score"}
    for source_name, generator in generators.items():
        try:
            df = generator(user_id=user_id, retrieval_k=retrieval_k)
        except Exception as exc:
            print(f"{source_name} failed " f"for user {user_id}: {exc}")
            continue
        if df.empty:
            continue
        score_column = score_mapping[source_name]
        df = df.copy()
        df["source"] = source_name
        df["source_rank"] = (df[score_column].rank(method="first",ascending=False).astype(int))
        output.append(df)
    if not output:
        return pd.DataFrame(columns=["consumer_id","item_id","source","source_rank"])
    return pd.concat(output,ignore_index=True)

## 45. Candidate Union

In [93]:
candidate_frames = []
for counter, user_id in enumerate(test_truth.keys(), start=1):
    candidates = generate_source_candidates(user_id=user_id, retrieval_k=CANDIDATE_K)
    if not candidates.empty:
        candidate_frames.append(candidates)
    if counter % 100 == 0:
        print(f"Processed " f"{counter}/{len(test_truth)}")
if candidate_frames:
    raw_candidates = pd.concat(candidate_frames,ignore_index=True)
else:
    raw_candidates = pd.DataFrame()
print("Raw candidates:", raw_candidates.shape)

Processed 100/1715
Processed 200/1715
Processed 300/1715
Processed 400/1715
Processed 500/1715
Processed 600/1715
Processed 700/1715
Processed 800/1715
Processed 900/1715
Processed 1000/1715
Processed 1100/1715
Processed 1200/1715
Processed 1300/1715
Processed 1400/1715
Processed 1500/1715
Processed 1600/1715
Processed 1700/1715
Raw candidates: (924719, 10)


# 46. Save Raw Candidates

In [94]:
raw_candidates_path = (CANDIDATE_DIR / "raw_candidates.parquet")
raw_candidates.to_parquet(raw_candidates_path,index=False)
print("Saved:", raw_candidates_path)

Saved: D:\iPrint-News-Recommendation-Ranking-System\artifacts\candidates\raw_candidates.parquet


## 47. Candidate Aggregation

In [97]:
print("raw_candidates shape:", raw_candidates.shape)
print("Columns:")
for col in raw_candidates.columns:
    print(" -", col)

raw_candidates shape: (924719, 10)
Columns:
 - consumer_id
 - item_id
 - popularity_score
 - source
 - source_rank
 - trending_score
 - country_score
 - tfidf_score
 - cf_score
 - als_score


In [99]:
expected_score_columns = ["popularity_score", "trending_score", "country_score", "tfidf_score","cf_score","als_score","semantic_score"]
missing_score_columns = [col for col in expected_score_columns if col not in raw_candidates.columns]
print("\nMissing score columns:")
print(missing_score_columns)


Missing score columns:
['semantic_score']


In [102]:
semantic_test = generate_semantic_candidates(sample_user,retrieval_k=10)
print(semantic_test)
print("\nColumns:")
print(semantic_test.columns.tolist())

Empty DataFrame
Columns: [consumer_id, item_id, semantic_score]
Index: []

Columns:
['consumer_id', 'item_id', 'semantic_score']


In [106]:
score_columns = ["popularity_score","trending_score","country_score","tfidf_score","cf_score","als_score","semantic_score"]
for col in score_columns:
    if col not in raw_candidates.columns:
        raw_candidates[col] = np.nan
candidate_scores = (raw_candidates.groupby(["consumer_id","item_id"],as_index=False)[score_columns].max())
print("Aggregated candidate scores:", candidate_scores.shape)
print(candidate_scores.head())

Aggregated candidate scores: (666597, 9)
            consumer_id               item_id  popularity_score  \
0  -1007001694607905623     -1038011342017850          0.811066   
1  -1007001694607905623  -1101361754763388054               NaN   
2  -1007001694607905623   -115909536143817330               NaN   
3  -1007001694607905623  -1199490911632553070               NaN   
4  -1007001694607905623  -1254906787526072320               NaN   

   trending_score  country_score  tfidf_score  cf_score  als_score  \
0             NaN            NaN          NaN       NaN        NaN   
1             NaN            NaN          NaN  0.677292        NaN   
2             NaN            NaN     0.186781       NaN        NaN   
3             NaN            NaN          NaN  0.664862        NaN   
4             NaN            NaN     0.127378       NaN        NaN   

   semantic_score  
0             NaN  
1             NaN  
2             NaN  
3             NaN  
4             NaN  


In [107]:
missing_score_columns = [col for col in score_columns  if col not in raw_candidates.columns]
if missing_score_columns:
    raise ValueError("Candidate generation is incomplete. " f"Missing score columns: {missing_score_columns}")

In [108]:
print(raw_candidates.columns.tolist())
print(generate_semantic_candidates(sample_user, retrieval_k=10).columns.tolist())
print("semantic_score" in raw_candidates.columns)

['consumer_id', 'item_id', 'popularity_score', 'source', 'source_rank', 'trending_score', 'country_score', 'tfidf_score', 'cf_score', 'als_score', 'semantic_score']
['consumer_id', 'item_id', 'semantic_score']
True


In [110]:
score_columns = ["popularity_score","trending_score","country_score","tfidf_score","cf_score","als_score","semantic_score"]
candidate_scores = (raw_candidates.groupby(["consumer_id","item_id"],as_index=False)[score_columns].max())

## 48. Source Count

In [113]:
source_count = (raw_candidates.groupby(["consumer_id","item_id"])["source"].nunique().reset_index(name="num_sources"))
candidate_scores = candidate_scores.merge(source_count,on=["consumer_id","item_id"],how="left")

In [114]:
source_flags_df = (raw_candidates[["consumer_id","item_id","source"]].drop_duplicates().assign(value=1).pivot_table(index=["consumer_id","item_id"],columns="source",values="value",fill_value=0).reset_index())
source_flags_df.columns.name = None
for source in ["popularity","trending","country","tfidf","item_cf","als","semantic"]:
    if source not in source_flags_df.columns:
        source_flags_df[source] = 0
    source_flags_df = source_flags_df.rename(columns={source:f"from_{source}"})
candidate_scores = candidate_scores.merge(source_flags_df,on=["consumer_id","item_id"],how="left")

## 49. Minimum Candidate Pool

In [116]:
candidate_count_per_user = (candidate_scores.groupby("consumer_id").size())
print(candidate_count_per_user.describe())
print("Users with >= 100 candidates:", (candidate_count_per_user >= 100).sum())
print("Users with >= 200 candidates:", (candidate_count_per_user >= 200).sum())
print("Users with >= 300 candidates:", (candidate_count_per_user >= 300).sum())

count    1715.000000
mean      388.686297
std        35.618909
min       244.000000
25%       378.000000
50%       396.000000
75%       412.000000
max       485.000000
dtype: float64
Users with >= 100 candidates: 1715
Users with >= 200 candidates: 1715
Users with >= 300 candidates: 1682


## 50. Candidate Recall Against Test Truth

In [118]:
def candidate_recall_at_k(candidates_df, truth_dict, k=None):
    recalls = []
    grouped = (candidates_df.groupby("consumer_id"))
    for user_id, truth_items in truth_dict.items():
        if not truth_items:
            continue
        if user_id not in grouped.groups:
            recalls.append(0.0)
            continue
        user_candidates = (grouped.get_group(user_id))
        if k is not None:
            user_candidates = (user_candidates.sort_values("candidate_priority",ascending=False).head(k))
        recommended_items = set(user_candidates["item_id"])
        hits = (recommended_items & set(truth_items))
        recalls.append(len(hits) / len(truth_items))
    if not recalls:
        return 0.0
    return float(np.mean(recalls))

In [120]:
candidate_sets = (candidate_scores.groupby("consumer_id")["item_id"].agg(set).to_dict())
recalls = []
for user_id, truth_items in test_truth.items():
    if not truth_items:
        continue
    candidates = candidate_sets.get(user_id,set())
    hit_count = len(candidates & truth_items)
    recalls.append(hit_count / len(truth_items))
raw_candidate_recall = (np.mean(recalls) if recalls else 0.0)
print("Raw candidate recall:", round(raw_candidate_recall,4))

Raw candidate recall: 0.4536


## 51. Source Level Recall

In [122]:
source_recall_results = []
for source_name in ["popularity","trending","country","tfidf","item_cf","als","semantic"]:
    source_df = raw_candidates[raw_candidates["source"] == source_name]
    source_sets = (source_df.groupby("consumer_id")["item_id"].agg(set).to_dict())
    source_recalls = []
    for user_id, truth_items in test_truth.items():
        candidates = source_sets.get(user_id, set())
        if not truth_items:
            continue
        source_recalls.append(len(candidates & truth_items) / len(truth_items))
    source_recall_results.append({"source": source_name,"recall": (np.mean(source_recalls)if source_recalls else 0.0)})
source_recall_df = pd.DataFrame(source_recall_results).sort_values("recall",ascending=False)
display(source_recall_df)

,source,recall
2,country,0.267055
5,als,0.246647
0,popularity,0.195918
1,trending,0.172595
4,item_cf,0.118950
3,tfidf,0.054810
6,semantic,0.000000


## 52. Candidate Priority

In [124]:
candidate_scores["candidate_priority"] = (0.20 * candidate_scores["popularity_score"] + 0.20 * candidate_scores["trending_score"] +  0.15 * candidate_scores["tfidf_score"] + 0.15 * candidate_scores["cf_score"] + 0.15 * candidate_scores["als_score"] + 0.15 * candidate_scores["semantic_score"])

## 53. Final Candidates Cap

In [126]:
candidate_scores = (candidate_scores.sort_values(["consumer_id","candidate_priority"],ascending=[True,False]))
final_candidates = (candidate_scores.groupby("consumer_id", group_keys=False).head(FINAL_CANDIDATE_K).reset_index(drop=True))
print("Final candidate rows:", len(final_candidates))
print("Final candidate users:", final_candidates["consumer_id"].nunique())

Final candidate rows: 514172
Final candidate users: 1715


## 54. Validate Seen Violations

In [127]:
seen_violations = []
for _, row in final_candidates.iterrows():
    seen = get_seen_items(row["consumer_id"])
    seen_violations.append(row["item_id"] in seen)
print("Seen violations:", sum(seen_violations))
assert sum(seen_violations) == 0

Seen violations: 0


In [128]:
availability_violations = (~final_candidates["item_id"].isin(available_items))
print("Availability violations:", availability_violations.sum())
assert availability_violations.sum() == 0

Availability violations: 0


In [129]:
duplicate_count = (final_candidates.duplicated(subset=["consumer_id","item_id"]).sum())
print("Duplicate user-item candidates:", duplicate_count)
assert duplicate_count == 0

Duplicate user-item candidates: 0


## 55. Candidate Distribution

In [131]:
candidate_distribution = (final_candidates.groupby("consumer_id").size())
display(candidate_distribution.describe(percentiles=[0.25,0.50,0.75,0.90,0.95,0.99]))

count    1715.000000
mean      299.808746
std         2.162488
min       244.000000
25%       300.000000
50%       300.000000
75%       300.000000
90%       300.000000
95%       300.000000
99%       300.000000
max       300.000000
dtype: float64

## 56. Source Overlap

In [133]:
source_overlap = (raw_candidates.groupby(["consumer_id","item_id"])["source"].nunique())
print("Candidates from exactly 1 source:", (source_overlap == 1).sum())
print("Candidates from 2+ sources:", (source_overlap >= 2).sum())
print("Candidates from 3+ sources:", (source_overlap >= 3).sum())

Candidates from exactly 1 source: 468467
Candidates from 2+ sources: 198130
Candidates from 3+ sources: 50505


## 57. Multi Source Candidates Percentages

In [139]:
multi_source_percentage = ((source_overlap >= 2).mean() * 100)
print("Multi-source candidate percentage:", round(multi_source_percentage, 2), "%")

Multi-source candidate percentage: 29.72 %


## 58. Candidates Features Tables

In [141]:
candidate_feature_columns = [
    # Keys
    "consumer_id", "item_id",
    # Retrieval scores
    "popularity_score","trending_score","country_score","tfidf_score","cf_score","als_score","semantic_score",
    # Retrieval source features
    "num_sources","from_popularity","from_trending","from_country","from_tfidf","from_item_cf","from_als","from_semantic",
    # Article metadata
    "producer_id","item_type","language","title","text_description","item_url","producer_country",
    # Article statistics
    "total_interactions","unique_users","weighted_interactions","log_total_interactions","log_unique_users","log_weighted_interactions",
    # Freshness
    "article_age_days","freshness_score",
    # User statistics
    "user_interactions","user_unique_items","user_unique_sessions","user_total_weight","user_inactivity_days","user_activity_recency",
    # Affinity
    "country_match","producer_interactions","producer_weight","recent_session_related",
    # Debug / fallback
    "candidate_priority",
]
candidate_feature_columns = [column for column in candidate_feature_columns if column in final_candidates.columns]
candidate_features = final_candidates[candidate_feature_columns].copy()
print("Candidate feature table:", candidate_features.shape)

Candidate feature table: (514172, 10)


## 59. Add Ranking Groups Informations

In [143]:
candidate_features = (candidate_features.sort_values(["consumer_id","candidate_priority"],ascending=[True,False]).reset_index(drop=True))
candidate_features["group_id"] = pd.factorize(candidate_features["consumer_id"])[0]
group_sizes = (candidate_features.groupby("group_id").size())
print("Number of ranking groups:", len(group_sizes))
print("Average candidates/group:", group_sizes.mean())

Number of ranking groups: 1715
Average candidates/group: 299.8087463556851


## 60. Generate Labels For Downstream Light Gradient Boosting Machines

In [145]:
candidate_features["label"] = (candidate_features.apply(lambda row: int(row["item_id"] in test_truth.get(row["consumer_id"],set())),axis=1))
print("Positive candidate rows:", candidate_features["label"].sum())
print("Positive rate:", candidate_features["label"].mean())

Positive candidate rows: 648
Positive rate: 0.0012602786616151794


## 61. Candidates Recall After Final Caps

In [147]:
final_candidate_sets = (candidate_features.groupby("consumer_id")["item_id"].agg(set).to_dict())
final_recalls = []
for user_id, truth_items in test_truth.items():
    if not truth_items:
        continue
    candidates = final_candidate_sets.get(user_id,set())
    final_recalls.append(len(candidates  & truth_items) / len(truth_items))
final_recall = (np.mean(final_recalls) if final_recalls else 0.0)
print( "Final candidate recall:", round(final_recall, 4))

Final candidate recall: 0.3778


## 62. Candidates Coverages

In [149]:
available_item_count = len(available_items)
candidate_item_count = (candidate_features["item_id"].nunique())
candidate_coverage = (candidate_item_count / available_item_count)
print("Available items:", available_item_count)
print("Candidate items:", candidate_item_count)
print("Candidate coverage:", round(candidate_coverage, 4)
)

Available items: 2983
Candidate items: 2901
Candidate coverage: 0.9725


## 63. Professional Recommenders Hanlde Users Without Enough History

In [152]:
all_consumer_users = set(consumer["consumer_id"])
training_users = set(train_consumer["consumer_id"])
cold_start_users = (all_consumer_users  - training_users)
print("Total users:", len(all_consumer_users))
print("Training users:", len(training_users))
print("Cold-start users:", len(cold_start_users))

Total users: 1895
Training users: 1715
Cold-start users: 180


In [153]:
def generate_cold_start_candidates(user_id,retrieval_k=100):
    frames = []
    trending = generate_trending_candidates(user_id=user_id,retrieval_k=retrieval_k)
    if not trending.empty:
        frames.append(trending)
    popularity = generate_popularity_candidates(user_id=user_id,retrieval_k=retrieval_k)
    if not popularity.empty:
        frames.append(popularity)
    country = generate_country_candidates(user_id=user_id,retrieval_k=retrieval_k)
    if not country.empty:
        frames.append(country)
    if not frames:
        return pd.DataFrame(columns=["consumer_id","item_id"])
    combined = pd.concat(frames,ignore_index=True)
    combined = (combined.drop_duplicates(subset=["consumer_id","item_id"]).head(retrieval_k))
    return combined.reset_index(drop=True)

## Save Candidates Dataset

In [155]:
candidate_features_path = (CANDIDATE_DIR / "candidate_features.parquet")
candidate_features.to_parquet(candidate_features_path, index=False)
print( "Saved:",candidate_features_path)

Saved: D:\iPrint-News-Recommendation-Ranking-System\artifacts\candidates\candidate_features.parquet


## Save CSV For Inspections

In [156]:
candidate_csv_path = (CANDIDATE_DIR / "candidate_features.csv")
candidate_features.to_csv(candidate_csv_path, index=False)
print("Saved:", candidate_csv_path)

Saved: D:\iPrint-News-Recommendation-Ranking-System\artifacts\candidates\candidate_features.csv


## Save Test Truth

In [158]:
test_truth_rows = []
for user_id, items in test_truth.items():
    for item_id in items:
        test_truth_rows.append({"consumer_id": user_id,"item_id": item_id})
test_truth_df = pd.DataFrame(test_truth_rows)
test_truth_path = (CANDIDATE_DIR / "test_truth.parquet")
test_truth_df.to_parquet(test_truth_path, index=False)
print("Saved:", test_truth_path)

Saved: D:\iPrint-News-Recommendation-Ranking-System\artifacts\candidates\test_truth.parquet


## Candidates Generation Summary

In [162]:
summary = {"users_evaluated": candidate_features["consumer_id"].nunique(),"available_articles":len(available_items),"candidate_rows":len(candidate_features),"unique_candidate_articles":candidate_features["item_id"].nunique(),"average_candidates_per_user":candidate_features.groupby("consumer_id").size().mean(),"candidate_recall":final_recall,"candidate_coverage":candidate_coverage,"seen_violations":int(sum(seen_violations)),"availability_violations":int(sum(availability_violations)),}
summary_df = pd.DataFrame([summary]).T.rename(columns={0: "value"})
display(summary_df)

,value
users_evaluated,1715.000000
available_articles,2983.000000
candidate_rows,514172.000000
unique_candidate_articles,2901.000000
average_candidates_per_user,299.808746
candidate_recall,0.377843
candidate_coverage,0.972511
seen_violations,0.000000
availability_violations,0.000000


## Inspect One Users Candidates Pools

In [181]:
required_display_columns = ["item_id","title","language","num_sources","popularity_score","trending_score","tfidf_score","cf_score","als_score","semantic_score","freshness_score","candidate_priority","label",]
missing_columns = [col for col in required_display_columns if col not in candidate_features.columns]
print("candidate_features shape:", candidate_features.shape)
print("Missing columns:")
print(missing_columns)
print("Available columns:")
print(candidate_features.columns.tolist())

candidate_features shape: (666597, 40)
Missing columns:
[]
Available columns:
['consumer_id', 'item_id', 'popularity_score', 'trending_score', 'country_score', 'tfidf_score', 'cf_score', 'als_score', 'semantic_score', 'num_sources_x', 'from_als_x', 'from_country_x', 'from_item_cf_x', 'from_popularity_x', 'from_tfidf_x', 'from_trending_x', 'from_semantic_x', 'num_sources_y', 'from_als_y', 'from_country_y', 'from_item_cf_y', 'from_popularity_y', 'from_tfidf_y', 'from_trending_y', 'from_semantic_y', 'candidate_priority', 'num_sources', 'event_datetime', 'title_article', 'language_article', 'item_type_article', 'producer_id_article', 'article_age_days', 'freshness_score', 'label', 'title', 'language', 'item_type', 'producer_id', 'article_published_at']


In [182]:
score_columns = ["popularity_score","trending_score","country_score","tfidf_score","cf_score","als_score","semantic_score",]
for col in score_columns:
    if col not in candidate_scores.columns:
        candidate_scores[col] = np.nan
candidate_scores["num_sources"] = (candidate_scores[score_columns].notna().sum(axis=1))
print(candidate_scores[["consumer_id","item_id","num_sources"]].head())

            consumer_id               item_id  num_sources
0  -1007001694607905623     -1038011342017850            1
1  -1007001694607905623  -1101361754763388054            1
2  -1007001694607905623   -115909536143817330            1
3  -1007001694607905623  -1199490911632553070            1
4  -1007001694607905623  -1254906787526072320            1


In [184]:
article_metadata = (content[["item_id","title","language","item_type","producer_id"]].drop_duplicates(subset=["item_id"]).copy())
article_metadata["item_id"] = (article_metadata["item_id"].astype(str).str.strip())
candidate_scores["item_id"] = (candidate_scores["item_id"].astype(str).str.strip())
candidate_scores = candidate_scores.merge(article_metadata,on="item_id",how="left")
print(candidate_scores[["item_id","title","language"]].head())

                item_id                                              title  \
0     -1038011342017850  Para entender o Dia Internacional contra a Hom...   
1  -1101361754763388054  Browser Trends December 2016: Mobile Overtakes...   
2   -115909536143817330  8 must-see sessions for application developers...   
3  -1199490911632553070                Our password hashing has no clothes   
4  -1254906787526072320  Enterprise developers look out: this week on G...   

  language  
0       pt  
1       en  
2       en  
3       en  
4       en  


In [185]:
article_catalog_features = (content[["item_id","event_datetime","title","language","item_type","producer_id"]].sort_values(["item_id","event_datetime"]).drop_duplicates(subset=["item_id"],keep="last").copy())
article_catalog_features["item_id"] = (article_catalog_features["item_id"].astype(str).str.strip())
candidate_scores = candidate_scores.merge(article_catalog_features,on="item_id",how="left",suffixes=("", "_article"))
print(candidate_scores[["item_id","event_datetime","title","language"]].head())

                item_id            event_datetime  \
0     -1038011342017850 2016-05-17 16:50:00+00:00   
1  -1101361754763388054 2016-12-07 12:42:59+00:00   
2   -115909536143817330 2017-02-03 13:18:04+00:00   
3  -1199490911632553070 2016-06-29 11:47:16+00:00   
4  -1254906787526072320 2016-06-02 16:11:44+00:00   

                                               title language  
0  Para entender o Dia Internacional contra a Hom...       pt  
1  Browser Trends December 2016: Mobile Overtakes...       en  
2  8 must-see sessions for application developers...       en  
3                Our password hashing has no clothes       en  
4  Enterprise developers look out: this week on G...       en  


In [186]:
reference_time = train_consumer["event_datetime"].max()
candidate_scores["article_age_days"] = ((reference_time - candidate_scores["event_datetime"]).dt.total_seconds() / 86400.0).clip(lower=0)
candidate_scores["freshness_score"] = (0.5 ** (candidate_scores["article_age_days"] / RECENCY_HALF_LIFE_DAYS))
print(candidate_scores[["item_id","article_age_days","freshness_score"]].head())

                item_id  article_age_days  freshness_score
0     -1038011342017850        287.085706     6.714941e-07
1  -1101361754763388054         83.257245     1.621029e-02
2   -115909536143817330         25.232882     2.867074e-01
3  -1199490911632553070        244.295938     5.586167e-06
4  -1254906787526072320        271.112280     1.480830e-06


In [187]:
candidate_features = candidate_scores.copy()
print("candidate_features shape:", candidate_features.shape)
print("Required columns available?")
for col in required_display_columns:
    print(f"{col:25s}:", col in candidate_features.columns)

candidate_features shape: (666597, 51)
Required columns available?
item_id                  : True
title                    : True
language                 : True
num_sources              : True
popularity_score         : True
trending_score           : True
tfidf_score              : True
cf_score                 : True
als_score                : True
semantic_score           : True
freshness_score          : True
candidate_priority       : True
label                    : False


In [188]:
print("candidate_features shape:", candidate_features.shape)
print("\nAvailable columns:")
print(candidate_features.columns.tolist())
required_columns = ["item_id","title","language","num_sources","popularity_score","trending_score","tfidf_score","cf_score","als_score","semantic_score","freshness_score","candidate_priority","label",]
missing_columns = [col for col in required_columns if col not in candidate_features.columns]
print("Missing columns:")
print(missing_columns)

candidate_features shape: (666597, 51)

Available columns:
['consumer_id', 'item_id', 'popularity_score', 'trending_score', 'country_score', 'tfidf_score', 'cf_score', 'als_score', 'semantic_score', 'num_sources_x', 'from_als_x', 'from_country_x', 'from_item_cf_x', 'from_popularity_x', 'from_tfidf_x', 'from_trending_x', 'from_semantic_x', 'num_sources_y', 'from_als_y', 'from_country_y', 'from_item_cf_y', 'from_popularity_y', 'from_tfidf_y', 'from_trending_y', 'from_semantic_y', 'candidate_priority', 'num_sources', 'title_x', 'language_x', 'item_type_x', 'producer_id_x', 'event_datetime', 'title_article', 'language_article', 'item_type_article', 'producer_id_article', 'article_age_days', 'freshness_score', 'title_y', 'language_y', 'item_type_y', 'producer_id_y', 'title', 'language', 'item_type', 'producer_id', 'event_datetime_article', 'title_article', 'language_article', 'item_type_article', 'producer_id_article']
Missing columns:
['label']


In [189]:
candidate_features["item_id"] = (candidate_features["item_id"].astype(str).str.strip())
test_truth_str = {str(user_id): {str(item_id).strip() for item_id in item_ids} for user_id, item_ids in test_truth.items()}
candidate_features["label"] = [int(str(item_id).strip() in test_truth_str.get(str(user_id), set())) for user_id, item_id in zip(candidate_features["consumer_id"],candidate_features["item_id"])]
print("Label distribution:")
print(candidate_features["label"].value_counts())
print("Positive labels:")
print(candidate_features["label"].sum())
print("Positive label rate:")
print(candidate_features["label"].mean())

Label distribution:
label
0    665819
1       778
Name: count, dtype: int64
Positive labels:
778
Positive label rate:
0.0011671219642452636


In [190]:
score_columns = ["popularity_score","trending_score","country_score","tfidf_score","cf_score","als_score","semantic_score",]
for col in score_columns:
    if col not in candidate_features.columns:
        candidate_features[col] = np.nan
candidate_features["num_sources"] = (candidate_features[score_columns].notna().sum(axis=1))
print(candidate_features["num_sources"].value_counts().sort_index())

num_sources
1    468467
2    147625
3     41935
4      7688
5       847
6        35
Name: count, dtype: int64


In [191]:
article_metadata = (content[["item_id","title","language","item_type","producer_id",]].drop_duplicates(subset=["item_id"]).copy())
article_metadata["item_id"] = (article_metadata["item_id"].astype(str).str.strip())
candidate_features["item_id"] = (candidate_features["item_id"].astype(str).str.strip())
metadata_columns = ["title","language","item_type","producer_id",]
candidate_features = candidate_features.drop(columns=[col for col in metadata_columns if col in candidate_features.columns],errors="ignore")
candidate_features = candidate_features.merge(article_metadata, on="item_id", how="left")
print(candidate_features[["item_id", "title", "language"]].head())

                item_id                                              title  \
0     -1038011342017850  Para entender o Dia Internacional contra a Hom...   
1  -1101361754763388054  Browser Trends December 2016: Mobile Overtakes...   
2   -115909536143817330  8 must-see sessions for application developers...   
3  -1199490911632553070                Our password hashing has no clothes   
4  -1254906787526072320  Enterprise developers look out: this week on G...   

  language  
0       pt  
1       en  
2       en  
3       en  
4       en  


In [192]:
content["event_timestamp"] = pd.to_numeric(content["event_timestamp"], errors="coerce")
content["event_datetime"] = pd.to_datetime(content["event_timestamp"],unit="s",utc=True,errors="coerce")
article_first_present = (content[content["interaction_type"].eq("content_present")].sort_values(["item_id", "event_datetime"]).drop_duplicates(subset=["item_id"],keep="first")[["item_id","event_datetime",]].rename(columns={"event_datetime": "article_published_at"}).copy())
article_first_present["item_id"] = (article_first_present["item_id"].astype(str).str.strip())
candidate_features = candidate_features.drop(columns=["article_published_at"],errors="ignore")
candidate_features = candidate_features.merge(article_first_present,on="item_id",how="left")
reference_time = train_consumer["event_datetime"].max()
candidate_features["article_age_days"] = ((reference_time - candidate_features["article_published_at"]).dt.total_seconds() / 86400.0).clip(lower=0)
RECENCY_HALF_LIFE_DAYS = 14
candidate_features["freshness_score"] = (0.5 ** (candidate_features["article_age_days"] / RECENCY_HALF_LIFE_DAYS))
print(candidate_features[["item_id","article_published_at","article_age_days","freshness_score",]].head())

                item_id      article_published_at  article_age_days  \
0     -1038011342017850 2016-05-17 16:50:00+00:00        287.085706   
1  -1101361754763388054 2016-12-07 12:42:59+00:00         83.257245   
2   -115909536143817330 2017-02-03 13:18:04+00:00         25.232882   
3  -1199490911632553070 2016-06-29 11:47:16+00:00        244.295938   
4  -1254906787526072320 2016-06-02 16:11:44+00:00        271.112280   

   freshness_score  
0     6.714941e-07  
1     1.621029e-02  
2     2.867074e-01  
3     5.586167e-06  
4     1.480830e-06  


In [193]:
if "candidate_priority" not in candidate_features.columns:
    candidate_features["candidate_priority"] = (candidate_features[["popularity_score","trending_score","country_score","tfidf_score","cf_score","als_score","semantic_score",]].fillna(0).sum(axis=1))
    if "freshness_score" in candidate_features.columns:
        candidate_features["candidate_priority"] += (candidate_features["freshness_score"])

In [194]:
inspect_user = sample_user
user_candidates = (candidate_features[candidate_features["consumer_id"] == inspect_user].sort_values("candidate_priority",ascending=False))
display(user_candidates[["item_id","title","language","num_sources","popularity_score","trending_score","tfidf_score","cf_score","als_score","semantic_score","freshness_score","candidate_priority","label",]].head(30))

,item_id,title,language,num_sources,popularity_score,trending_score,tfidf_score,cf_score,als_score,semantic_score,freshness_score,candidate_priority,label
0,-1038011342017850,Para entender o Dia Internacional contra a Hom...,pt,1,0.811066,NaN,NaN,NaN,NaN,NaN,6.714941e-07,NaN,0
1,-1101361754763388054,Browser Trends December 2016: Mobile Overtakes...,en,1,NaN,NaN,NaN,0.677292,NaN,NaN,1.621029e-02,NaN,0
2,-115909536143817330,8 must-see sessions for application developers...,en,1,NaN,NaN,0.186781,NaN,NaN,NaN,2.867074e-01,NaN,0
3,-1199490911632553070,Our password hashing has no clothes,en,1,NaN,NaN,NaN,0.664862,NaN,NaN,5.586167e-06,NaN,0
4,-1254906787526072320,Enterprise developers look out: this week on G...,en,1,NaN,NaN,0.127378,NaN,NaN,NaN,1.480830e-06,NaN,0
5,-1291869519847635026,Learning How to Learn: The Most Important Deve...,en,1,NaN,NaN,0.154672,NaN,NaN,NaN,1.094188e-04,NaN,0
6,-1297580205670251233,A minha viagem à Maternidade #tetodomundo,pt,2,0.963692,NaN,NaN,NaN,NaN,NaN,1.017826e-05,NaN,0
7,-133139342397538859,"Novo workaholic trabalha, pratica esportes e t...",pt,3,0.980638,NaN,NaN,NaN,0.037686,NaN,7.936751e-06,NaN,0
8,-1443900796086927361,What is IT's role in digital transformation?,en,2,NaN,NaN,NaN,0.642193,0.029114,NaN,3.469924e-04,NaN,0
9,-1444481301861872291,Nova campanha da Avon tem drag queen como estrela,pt,1,NaN,NaN,NaN,NaN,0.040943,NaN,2.608035e-07,NaN,0


In [195]:
required_columns = ["consumer_id","item_id","title","language","num_sources","popularity_score","trending_score","tfidf_score","cf_score","als_score","semantic_score","freshness_score","candidate_priority","label",]
missing = [col for col in required_columns if col not in candidate_features.columns]
if missing:
    raise ValueError(f"candidate_features is missing required columns: {missing}")
print("All candidate feature columns are available")
print(f" Shape: {candidate_features.shape}")
print(f"Positive labels: "f"{candidate_features['label'].sum()}")
print(f" Positive rate: "f"{candidate_features['label'].mean():.4%}")

All candidate feature columns are available
 Shape: (666597, 53)
Positive labels: 778
 Positive rate: 0.1167%


## Inspect Users Actual Test Items

In [197]:
print("User:", inspect_user)
print("Actual test item(s):", test_truth.get(inspect_user,set()))
test_items = test_truth.get(inspect_user,set())
display(article_catalog[article_catalog["item_id"].isin(test_items)][["item_id","title","language","item_type","producer_id"]])

User: -1007001694607905623
Actual test item(s): {'8729086959762650511'}


,item_id,title,language,item_type,producer_id
2941,8729086959762650511,"What do you mean by ""Event-Driven""?",en,HTML,7645894863578715801


## Check Whetther Test Item Entered Candidates Pools

In [199]:
for item_id in test_truth.get(inspect_user, set()):
    match = candidate_features[(candidate_features["consumer_id"] == inspect_user) & (candidate_features["item_id"] == item_id)]
    print(f"Item {item_id}:", "FOUND" if not match.empty else "NOT FOUND")

Item 8729086959762650511: FOUND


## Save Configurations

In [200]:
import json
candidate_config = {"candidate_k_per_source": CANDIDATE_K, "final_candidate_k": FINAL_CANDIDATE_K, "top_k": TOP_K,"recent_days":RECENT_DAYS,"recency_half_life_days":RECENCY_HALF_LIFE_DAYS,"interaction_weights":INTERACTION_WEIGHTS,"candidate_sources": ["popularity","trending","country","tfidf","item_cf","als","semantic",],"content_based_language":"en","availability_rule":"latest_content_state","seen_item_rule": "remove_training_history_items","split_strategy": "last_interaction_per_user",}
config_path = (CANDIDATE_DIR / "candidate_generation_config.json")
with open(config_path,"w", encoding="utf-8") as file:
    json.dump(candidate_config, file, indent=4, default=str)
print("Saved:", config_path)

Saved: D:\iPrint-News-Recommendation-Ranking-System\artifacts\candidates\candidate_generation_config.json
